# Data Preparation Pipeline for Turkish Legal RAG

This notebook demonstrates **Prompt 2: Data Loading and Cleaning**

## Goals
1. Load Turkish legal QA datasets from CSV/JSON
2. Normalize Turkish text (preserve Turkish characters: ç, ğ, ı, ö, ş, ü)
3. Remove duplicates
4. Preserve metadata fields
5. Save cleaned outputs as JSONL
6. Logging and error handling

## Data Sources
- **data/raw/example_lawchatbot.json**: HuggingFace format (question, answer, category)
- **data/raw/example_kaggle_turkishlaw.json**: Kaggle format (instruction, input, output)
- **data/raw/sample_qa_dataset.json**: Nested QA format
- **data/raw/sample_legal_texts.json**: Legal document format
- **data/raw/turkish_law_dataset.csv**: Large CSV (100MB) with Turkish column names

## Output
- **data/processed/**: Clean JSONL files ready for retrieval
- **Statistics**: Record counts, deduplication summary, schema compliance

## 1. Setup and Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import json
import csv
import logging
from pathlib import Path
from typing import List, Dict, Any, Optional
import hashlib
from collections import defaultdict

import pandas as pd
import numpy as np

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('DataPreparation')

print("✅ Imports successful")

## 2. Turkish Text Normalization

In [ ]:
def normalize_turkish_text(text: str, preserve_case: bool = False) -> str:
    """
    Normalize Turkish text while preserving Turkish characters.
    
    Turkish characters to preserve:
    - Lowercase: ç, ğ, ı, ö, ş, ü
    - Uppercase: Ç, Ğ, İ, Ö, Ş, Ü
    
    Normalizations:
    1. Remove control characters
    2. Normalize whitespace (multiple spaces → single space)
    3. Normalize quotes (curly → straight)
    4. Strip leading/trailing whitespace
    """
    if not isinstance(text, str):
        return ""
    
    # Remove control characters (but keep Turkish chars)
    text = ''.join(char for char in text if ord(char) >= 32 or char in '\\n\\t')
    
    # Normalize whitespace
    text = ' '.join(text.split())
    
    # Normalize quotes
    text = text.replace('
').replace('
')
    text = text.replace(''', "'").replace(''', "'")
    
    # Optional: lowercase
    if not preserve_case:
        text = text.lower()
    
    return text.strip()

# Test Turkish normalization
test_text = "  Türkiye'de   hukuk    sistemi  nasıl  çalışır?  "
normalized = normalize_turkish_text(test_text)
print(f"Original:   '{test_text}'")
print(f"Normalized: '{normalized}'")
print(f"✅ Turkish characters preserved: {all(c in normalized for c in 'çğıöşü')}")

## 3. Data Loading Functions

In [ ]:
def load_json_file(filepath: str) -> List[Dict[str, Any]]:
    """Load JSON file and handle nested structures."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Handle nested 'data' field
        if isinstance(data, dict) and 'data' in data:
            data = data['data']
        
        if not isinstance(data, list):
            data = [data]
        
        logger.info(f"Loaded {len(data)} records from {filepath}")
        return data
    except Exception as e:
        logger.error(f"Error loading JSON file {filepath}: {e}")
        return []

def load_jsonl_file(filepath: str) -> List[Dict[str, Any]]:
    """Load JSONL file (JSON Lines format)."""
    data = []
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                try:
                    data.append(json.loads(line))
                except json.JSONDecodeError as e:
                    logger.warning(f"Skipping line {line_num}: {e}")
        logger.info(f"Loaded {len(data)} records from {filepath}")
    except Exception as e:
        logger.error(f"Error loading JSONL file {filepath}: {e}")
    return data

def load_csv_file(filepath: str, encoding: str = 'utf-8') -> List[Dict[str, Any]]:
    """Load CSV file."""
    data = []
    try:
        df = pd.read_csv(filepath, encoding=encoding)
        data = df.to_dict('records')
        logger.info(f"Loaded {len(data)} records from {filepath}")
        logger.info(f"Columns: {list(df.columns)}")
    except Exception as e:
        logger.error(f"Error loading CSV file {filepath}: {e}")
    return data

def load_data(filepath: str) -> List[Dict[str, Any]]:
    """Auto-detect format and load data."""
    ext = Path(filepath).suffix.lower()
    
    if ext == '.json':
        return load_json_file(filepath)
    elif ext == '.jsonl':
        return load_jsonl_file(filepath)
    elif ext == '.csv':
        return load_csv_file(filepath)
    else:
        logger.error(f"Unsupported file format: {ext}")
        return []

# Test loading
print("\n--- Testing Data Loading ---")
example_qa = load_data('../data/raw/example_lawchatbot.json')
print(f"Loaded {len(example_qa)} example QA records")
if example_qa:
    print(f"First record: {example_qa[0]}")

## 4. Schema Conversion

In [ ]:
import uuid

def convert_to_qa_schema(records: List[Dict], source: str, field_mapping: Optional[Dict] = None) -> List[Dict]:
    """
    Convert records to unified QA schema.
    
    Standard QA Schema:
    {
        "id": "uuid",
        "question": "Turkish question",
        "answer": "Turkish answer",
        "source": "source_name",
        "category": "optional category",
        "citation": "optional citation"
    }
    """
    # Default field mapping
    if field_mapping is None:
        field_mapping = {
            'question': ['question', 'soru', 'q', 'instruction'],
            'answer': ['answer', 'cevap', 'a', 'output'],
            'category': ['category', 'kategori', 'type', 'veri türü'],
        }
    
    converted = []
    for record in records:
        try:
            # Auto-detect fields
            question = None
            answer = None
            
            for field_variants in field_mapping.values():
                for variant in field_variants:
                    if variant in record:
                        if field_mapping.get('question')[0] == variant.split('_')[0]:
                            question = record[variant]
                        elif field_mapping.get('answer')[0] == variant.split('_')[0]:
                            answer = record[variant]
            
            # Fallback: try common patterns
            if question is None:
                for key in record:
                    if any(q in key.lower() for q in ['question', 'soru', 'prompt']):
                        question = record[key]
                        break
            
            if answer is None:
                for key in record:
                    if any(a in key.lower() for a in ['answer', 'cevap', 'output']):
                        answer = record[key]
                        break
            
            if question and answer:
                converted.append({
                    "id": str(uuid.uuid4()),
                    "question": normalize_turkish_text(str(question)),
                    "answer": normalize_turkish_text(str(answer)),
                    "source": source,
                    "category": str(record.get('category', record.get('kategori', record.get('type', '')))).strip(),
                    "citation": ""
                })
        except Exception as e:
            logger.warning(f"Skipping record: {e}")
    
    logger.info(f"Converted {len(converted)} records")
    return converted

# Test conversion
print("\n--- Testing Schema Conversion ---")
converted = convert_to_qa_schema(example_qa, "example_lawchatbot")
if converted:
    print(f"Converted {len(converted)} records")
    print(f"Sample: {json.dumps(converted[0], ensure_ascii=False, indent=2)}")

## 5. Deduplication

In [ ]:
def deduplicate_records(records: List[Dict], dedup_field: str = 'question') -> tuple:
    """
    Remove duplicate records based on a field (case-insensitive hash).
    
    Returns: (deduplicated_records, duplicate_count)
    """
    seen_hashes = set()
    unique_records = []
    duplicates = 0
    
    for record in records:
        # Create hash of the dedup field
        field_value = str(record.get(dedup_field, '')).lower()
        content_hash = hashlib.md5(field_value.encode()).hexdigest()
        
        if content_hash not in seen_hashes:
            seen_hashes.add(content_hash)
            unique_records.append(record)
        else:
            duplicates += 1
    
    logger.info(f"Deduplication: {len(records)} → {len(unique_records)} records ({duplicates} duplicates removed)")
    return unique_records, duplicates

# Test deduplication
print("\n--- Testing Deduplication ---")
dedup_records, dup_count = deduplicate_records(converted)
print(f"Original: {len(converted)}, After dedup: {len(dedup_records)}, Removed: {dup_count}")

## 6. Save to JSONL

In [ ]:
def save_jsonl(records: List[Dict], output_path: str) -> bool:
    """
    Save records to JSONL format (JSON Lines).
    One JSON object per line.
    """
    try:
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_path, 'w', encoding='utf-8') as f:
            for record in records:
                f.write(json.dumps(record, ensure_ascii=False) + '\\n')
        
        logger.info(f"Saved {len(records)} records to {output_path}")
        return True
    except Exception as e:
        logger.error(f"Error saving JSONL: {e}")
        return False

# Test saving
print("\n--- Testing JSONL Output ---")
output_file = '../data/processed/example_lawchatbot_cleaned.jsonl'
success = save_jsonl(dedup_records, output_file)
if success:
    file_size = Path(output_file).stat().st_size
    print(f"✅ Saved {len(dedup_records)} records to {output_file}")
    print(f"File size: {file_size / 1024:.2f} KB")

## 7. Full Pipeline: Load → Normalize → Deduplicate → Save

In [ ]:
def process_dataset(input_path: str, output_name: str, source: str, data_type: str = 'qa', 
                   field_mapping: Optional[Dict] = None) -> Dict:
    """
    Full data preparation pipeline.
    """
    logger.info(f"Starting pipeline for {input_path}")
    
    stats = {
        'input_file': input_path,
        'output_file': f'data/processed/{output_name}.jsonl',
        'loaded_count': 0,
        'converted_count': 0,
        'duplicates_removed': 0,
        'final_count': 0,
        'source': source
    }
    
    # 1. Load
    raw_data = load_data(input_path)
    stats['loaded_count'] = len(raw_data)
    
    if not raw_data:
        logger.error("No data loaded, aborting")
        return stats
    
    # 2. Convert
    converted_data = convert_to_qa_schema(raw_data, source, field_mapping)
    stats['converted_count'] = len(converted_data)
    
    # 3. Deduplicate
    unique_data, dups = deduplicate_records(converted_data)
    stats['duplicates_removed'] = dups
    stats['final_count'] = len(unique_data)
    
    # 4. Save
    output_path = f'../data/processed/{output_name}.jsonl'
    save_jsonl(unique_data, output_path)
    
    logger.info(f"Pipeline complete: {stats}")
    return stats

# Run pipeline on example datasets
print("\n" + "="*60)
print("RUNNING FULL DATA PREPARATION PIPELINE")
print("="*60)

all_stats = []

# Example 1: HuggingFace format
stats1 = process_dataset(
    '../data/raw/example_lawchatbot.json',
    'example_lawchatbot_cleaned',
    'example_lawchatbot'
)
all_stats.append(stats1)

# Example 2: Kaggle format
stats2 = process_dataset(
    '../data/raw/example_kaggle_turkishlaw.json',
    'example_kaggle_cleaned',
    'example_kaggle'
)
all_stats.append(stats2)

# Example 3: Sample legal texts
stats3 = process_dataset(
    '../data/raw/sample_legal_texts.json',
    'sample_legal_texts_cleaned',
    'sample_legal_texts'
)
all_stats.append(stats3)

## 8. Statistics Summary

In [ ]:
# Create summary
print("\n" + "="*80)
print("DATA PREPARATION PIPELINE SUMMARY")
print("="*80)

summary_df = pd.DataFrame(all_stats)
print(summary_df.to_string(index=False))

print(f"\n✅ Total Records Processed: {summary_df['final_count'].sum()}")
print(f"✅ Total Duplicates Removed: {summary_df['duplicates_removed'].sum()}")
print(f"✅ Output Location: data/processed/")

# Save summary
summary_df.to_csv('../data/processed/preparation_summary.csv', index=False)
logger.info("Summary saved to data/processed/preparation_summary.csv")

## 9. Verify Output Quality

In [ ]:
# Sample inspection
print("\n--- Sample Output Inspection ---")
sample_file = '../data/processed/example_lawchatbot_cleaned.jsonl'

if Path(sample_file).exists():
    # Read first few lines
    with open(sample_file, 'r', encoding='utf-8') as f:
        lines = [f.readline() for _ in range(3)]
    
    for i, line in enumerate(lines, 1):
        if line:
            record = json.loads(line)
            print(f"\nRecord {i}:")
            print(f"  ID: {record['id']}")
            print(f"  Question: {record['question'][:60]}...")
            print(f"  Answer: {record['answer'][:60]}...")
            print(f"  Category: {record['category']}")
            print(f"  Source: {record['source']}")
            
            # Check Turkish characters
            turkish_chars = ['ç', 'ğ', 'ı', 'ö', 'ş', 'ü', 'Ç', 'Ğ', 'İ', 'Ö', 'Ş', 'Ü']
            found_chars = [c for c in turkish_chars if c in record['question'] or c in record['answer']]
            if found_chars:
                print(f"  ✅ Turkish chars preserved: {found_chars}")
    
    print("\n✅ Sample output looks good!")
else:
    print(f"❌ Output file not found: {sample_file}")

## 10. Next Steps

✅ **Prompt 2 Complete**: Data preparation pipeline working!

### What's Next?

1. **Prompt 3: Document Chunking** 
   - Split long documents into retrieval chunks
   - Preserve metadata (article numbers, sections)
   - Generate chunk statistics

2. **Prompt 4: Dense Retrieval**
   - Use sentence-transformers to encode chunks
   - Build FAISS index
   - Implement semantic search

3. **Prompt 5: Hybrid Retrieval**
   - Add BM25 sparse search
   - Combine dense + sparse results
   - Compare retrieval methods

And continue through all 14 prompts...

---

**For more information, see:**
- [INGESTION.md](../INGESTION.md) — Detailed data loading guide
- [README.md](../README.md) — Project overview
- [HEALTH_CHECK_REPORT.md](../HEALTH_CHECK_REPORT.md) — System status